# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [2]:
# Carregando a base de dados 
df = pd.read_csv('Cientista de dados M35 - smoke_detection_iot.csv')

# Visualizando os primeiros dados do dataframe
df.head()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [3]:
# Verificando as colunas e tipos de dados
print("\nInformações do DataFrame:")
df.info()


Informações do DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      62630 non-null  int64  
 1   UTC             62630 non-null  int64  
 2   Temperature[C]  62630 non-null  float64
 3   Humidity[%]     62630 non-null  float64
 4   TVOC[ppb]       62630 non-null  int64  
 5   eCO2[ppm]       62630 non-null  int64  
 6   Raw H2          62630 non-null  int64  
 7   Raw Ethanol     62630 non-null  int64  
 8   Pressure[hPa]   62630 non-null  float64
 9   PM1.0           62630 non-null  float64
 10  PM2.5           62630 non-null  float64
 11  NC0.5           62630 non-null  float64
 12  NC1.0           62630 non-null  float64
 13  NC2.5           62630 non-null  float64
 14  CNT             62630 non-null  int64  
 15  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(8)
memory usage: 7.6 MB


OBS: Observando o retorno da função, ficou claro que não existem dados faltantes ou nulos neste dataframe, por este motivo, não será realizado nenhuma tratativa dos dados 

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.




Resposta: O modelo para essa tarefa deve ser de classificação, pois a nossa variável alvo (`Fire Alarm`) é binária. Cmo naõa estamos tentando prever uma numero continuo como temperatura ou preço, ma sum uma classe o modelo de classificação é mais indicado.

Por esse motivo, vamos aplicar a regressão logística 

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

Neste contexto, vamos remover os dados a seguir 
- `Unnamed: 0`: É apenas um indice e como já foi informado, não é util para o modelo
- `UTC`: É um horario e pode afetar o aprendizado do modelo, gravando as horas que o incêndio em vez de aprender a causa.
- `CNT`: Tratando-se de um contador, ele aumenta sequencialmente e pode vicar o modelo.

In [4]:
# Definindo a variável alvo (Target)
y = df['Fire Alarm']

# Definindo as variáveis explicativas (Features)
X = df.drop(columns=['Fire Alarm', 'Unnamed: 0', 'UTC', 'CNT'])

# # Instanciando o modelo de Regressão Logística
modelo = LogisticRegression(max_iter=1000)

print(f"Dimensões de X: {X.shape}")
print(f"Dimensões de y: {y.shape}")

Dimensões de X: (62630, 12)
Dimensões de y: (62630,)


# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [5]:
# Definindo o número de folds para a validação cruzada
num_folds = 5

# Rodando o cross_val_score
pontuacoes = cross_val_score(modelo, X, y, cv=num_folds)

# Exibindo os resultados de cada fold
print(f"Pontuações (Acurácia) em cada fold: {pontuacoes}")

Pontuações (Acurácia) em cada fold: [0.5171643  0.77534728 0.62454095 0.97804567 0.89837139]


c:\Users\Rafael\Dev\ciencia_de_dados_v2\venv3_14\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [6]:
# Calculando a média das pontuações
media_pontuacao = pontuacoes.mean()

print(f"Acurácia média do modelo: {media_pontuacao:.2%}")

Acurácia média do modelo: 75.87%
